In [ ]:
!pip install -q -U datasets transformers scikit-learn pandas matplotlib

import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

_probe = load_dataset("SetFit/amazon_reviews_multi_en", split="train[:1]")
print("Data access OK:", _probe[0]["text"][:80])

In [ ]:
SEEDS = [42]
TRAIN_SIZE = 2000
VAL_SIZE = 500
TEST_SIZE = 500

CHECKPOINTS = {
    "monolingual_en": "bert-base-uncased",
    "monolingual_de": "dbmdz/bert-base-german-cased",   # <-- changed from deepset/gbert-base
    "multilingual":   "bert-base-multilingual-cased",
}

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 2
LEARNING_RATE = 2e-5
PATIENCE = 1
OUTPUT_DIR = "/kaggle/working/"

In [ ]:
def load_marc_split(language, split):
    ds = load_dataset(f"SetFit/amazon_reviews_multi_{language}", split=split)
    texts = [row["text"] for row in ds]
    stars = [int(row["label"]) + 1 for row in ds]
    return texts, stars

def binarize_labels(texts, stars):
    kept_texts, kept_labels = [], []
    for text, star in zip(texts, stars):
        if star in (1, 2):
            kept_texts.append(text)
            kept_labels.append(0)
        elif star in (4, 5):
            kept_texts.append(text)
            kept_labels.append(1)
    return kept_texts, kept_labels

def stratified_subsample(texts, labels, n_examples, seed=42):
    if n_examples >= len(texts):
        return texts, labels
    subset_texts, _, subset_labels, _ = train_test_split(
        texts, labels, train_size=n_examples, random_state=seed, stratify=labels,
    )
    return subset_texts, subset_labels

def load_and_prepare(language, train_size, val_size, test_size, seed=42):
    result = {}
    sizes = {"train": train_size, "validation": val_size, "test": test_size}
    for split, size in sizes.items():
        texts, stars = load_marc_split(language, split)
        texts, labels = binarize_labels(texts, stars)
        texts, labels = stratified_subsample(texts, labels, size, seed=seed)
        result[split] = (texts, labels)
        pct_positive = 100 * np.mean(labels) if labels else 0.0
        print(f"[{language}] {split}: {len(texts)} examples ({pct_positive:.1f}% positive)")
    return result

print("Loading English data...")
en_data = load_and_prepare("en", TRAIN_SIZE, VAL_SIZE, TEST_SIZE, seed=42)
print("Loading German data...")
de_data = load_and_prepare("de", TRAIN_SIZE, VAL_SIZE, TEST_SIZE, seed=42)

en_train_texts, en_train_labels = en_data["train"]
en_val_texts, en_val_labels = en_data["validation"]
en_test_texts, en_test_labels = en_data["test"]

de_train_texts, de_train_labels = de_data["train"]
de_val_texts, de_val_labels = de_data["validation"]
de_test_texts, de_test_labels = de_data["test"]

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx], truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt",
        )
        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return input_ids, attention_mask, label

In [ ]:
def run_baseline(train_texts, train_labels, test_texts, test_labels, seed=42):
    vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)
    clf = LogisticRegression(max_iter=1000, random_state=seed)
    clf.fit(X_train, train_labels)
    preds = clf.predict(X_test)
    return {
        "test_accuracy": accuracy_score(test_labels, preds),
        "test_macro_f1": f1_score(test_labels, preds, average="macro"),
    }

results = []

baseline_en = run_baseline(en_train_texts, en_train_labels, en_test_texts, en_test_labels)
results.append({"condition": "baseline_en", "seed": 42, **baseline_en})
print("baseline_en:", baseline_en)

baseline_de = run_baseline(de_train_texts, de_train_labels, de_test_texts, de_test_labels)
results.append({"condition": "baseline_de", "seed": 42, **baseline_de})
print("baseline_de:", baseline_de)

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, checkpoint_name, num_classes=2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(checkpoint_name)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_embedding)

def train_one_epoch(model, data_loader, criterion, optimizer, device):
    model.train()
    total_loss, total_correct, total_examples = 0.0, 0, 0
    for input_ids, attention_mask, labels in data_loader:
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        preds = torch.argmax(logits, dim=1)
        total_loss += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total_examples += labels.size(0)
    return total_loss / total_examples, total_correct / total_examples

def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss, total_examples = 0.0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in data_loader:
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            preds = torch.argmax(logits, dim=1)
            total_loss += loss.item() * labels.size(0)
            total_examples += labels.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    accuracy = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return total_loss / total_examples, accuracy, macro_f1

In [ ]:
def run_condition(checkpoint_name, train_texts, train_labels, val_texts, val_labels,
                   test_texts, test_labels, seed=42):
    set_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    tokenizer = AutoTokenizer.from_pretrained(checkpoint_name)
    model = SentimentClassifier(checkpoint_name).to(device)

    train_loader = DataLoader(ReviewDataset(train_texts, train_labels, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(ReviewDataset(val_texts, val_labels, tokenizer), batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(ReviewDataset(test_texts, test_labels, tokenizer), batch_size=BATCH_SIZE, shuffle=False)

    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

    best_val_f1 = -1.0
    best_state = None
    epochs_without_improvement = 0

    start_time = time.perf_counter()
    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion, device)
        print(f"  epoch {epoch+1}/{EPOCHS} | train acc {train_acc:.3f} | val acc {val_acc:.3f} f1 {val_f1:.3f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print("  early stopping")
                break

    elapsed_seconds = time.perf_counter() - start_time
    if best_state is not None:
        model.load_state_dict(best_state)

    peak_memory_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if torch.cuda.is_available() else None
    test_loss, test_acc, test_f1 = evaluate(model, test_loader, criterion, device)

    return {
        "val_macro_f1": best_val_f1,
        "test_accuracy": test_acc,
        "test_macro_f1": test_f1,
        "train_seconds": elapsed_seconds,
        "peak_memory_mb": peak_memory_mb,
        "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    }

In [ ]:
conditions = [
    ("monolingual_en", CHECKPOINTS["monolingual_en"],
     en_train_texts, en_train_labels, en_val_texts, en_val_labels, en_test_texts, en_test_labels),
    ("monolingual_de", CHECKPOINTS["monolingual_de"],
     de_train_texts, de_train_labels, de_val_texts, de_val_labels, de_test_texts, de_test_labels),
    ("multilingual_in_language_en", CHECKPOINTS["multilingual"],
     en_train_texts, en_train_labels, en_val_texts, en_val_labels, en_test_texts, en_test_labels),
    ("multilingual_in_language_de", CHECKPOINTS["multilingual"],
     de_train_texts, de_train_labels, de_val_texts, de_val_labels, de_test_texts, de_test_labels),
    ("multilingual_zero_shot_transfer", CHECKPOINTS["multilingual"],
     en_train_texts, en_train_labels, en_val_texts, en_val_labels, de_test_texts, de_test_labels),
]

for (name, checkpoint, train_texts, train_labels, val_texts, val_labels, test_texts, test_labels) in conditions:
    for seed in SEEDS:
        print(f"\n{name} (checkpoint={checkpoint}, seed={seed})")
        metrics = run_condition(checkpoint, train_texts, train_labels, val_texts, val_labels,
                                 test_texts, test_labels, seed=seed)
        results.append({"condition": name, "seed": seed, **metrics})

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR + "results.csv", index=False)
print(results_df)

In [ ]:
#Additional seeds added after testing of og seed that works
extra_seeds = [43, 44]

core_conditions = [
    ("monolingual_de", CHECKPOINTS["monolingual_de"],
     de_train_texts, de_train_labels, de_val_texts, de_val_labels, de_test_texts, de_test_labels),
    ("multilingual_in_language_de", CHECKPOINTS["multilingual"],
     de_train_texts, de_train_labels, de_val_texts, de_val_labels, de_test_texts, de_test_labels),
    ("multilingual_zero_shot_transfer", CHECKPOINTS["multilingual"],
     en_train_texts, en_train_labels, en_val_texts, en_val_labels, de_test_texts, de_test_labels),
]

for (name, checkpoint, train_texts, train_labels, val_texts, val_labels, test_texts, test_labels) in core_conditions:
    for seed in extra_seeds:
        print(f"\n{name} (checkpoint={checkpoint}, seed={seed})")
        metrics = run_condition(checkpoint, train_texts, train_labels, val_texts, val_labels,
                                 test_texts, test_labels, seed=seed)
        results.append({"condition": name, "seed": seed, **metrics})

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR + "results.csv", index=False)
print("\nUpdated results saved.")
print(results_df[results_df["condition"].isin([c[0] for c in core_conditions])])

In [ ]:
# Pull real misclassified examples from the zero-shot condition (seed 42)
set_seed(42)
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINTS["multilingual"])
model = SentimentClassifier(CHECKPOINTS["multilingual"]).to(device)

train_loader = DataLoader(ReviewDataset(en_train_texts, en_train_labels, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ReviewDataset(en_val_texts, en_val_labels, tokenizer), batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

best_val_f1, best_state = -1.0, None
for epoch in range(EPOCHS):
    train_one_epoch(model, train_loader, criterion, optimizer, device)
    _, _, val_f1 = evaluate(model, val_loader, criterion, device)
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
model.eval()

# Run on German test set, collect mismatches
mismatches = []
with torch.no_grad():
    for i in range(len(de_test_texts)):
        enc = tokenizer(de_test_texts[i], truncation=True, max_length=MAX_LENGTH,
                         padding="max_length", return_tensors="pt")
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)
        logit = model(input_ids, attention_mask)
        pred = torch.argmax(logit, dim=1).item()
        true = de_test_labels[i]
        if pred != true:
            mismatches.append({"text": de_test_texts[i], "true": true, "pred": pred})

print(f"Total mismatches: {len(mismatches)} out of {len(de_test_texts)}")
label_names = {0: "negative", 1: "positive"}
for m in mismatches[:10]:
    print(f"\nTrue: {label_names[m['true']]} | Predicted: {label_names[m['pred']]}")
    print(m["text"][:200])

In [ ]:
from collections import Counter
direction_counts = Counter((m["true"], m["pred"]) for m in mismatches)
print(direction_counts)

In [ ]:
CONDITION_ORDER = [
    "baseline_en", "baseline_de", "monolingual_en", "monolingual_de",
    "multilingual_in_language_en", "multilingual_in_language_de", "multilingual_zero_shot_transfer",
]
CONDITION_LABELS = {
    "baseline_en": "Baseline (TF-IDF+LogReg), EN",
    "baseline_de": "Baseline (TF-IDF+LogReg), DE",
    "monolingual_en": "Monolingual EN (BERT)",
    "monolingual_de": "Monolingual DE (GBERT)",
    "multilingual_in_language_en": "Multilingual EN (mBERT)",
    "multilingual_in_language_de": "Multilingual DE (mBERT)",
    "multilingual_zero_shot_transfer": "Zero-shot transfer (mBERT)",
}

results_df["condition"] = pd.Categorical(results_df["condition"], categories=CONDITION_ORDER, ordered=True)
results_df = results_df.sort_values("condition")

summary = results_df.groupby("condition", observed=True)[["test_accuracy", "test_macro_f1"]].agg(["mean", "std"])
summary.columns = ["_".join(col) for col in summary.columns]
summary = summary.reset_index()

cost_summary = results_df.groupby("condition", observed=True)[
    ["train_seconds", "peak_memory_mb", "trainable_params"]
].mean().reset_index()

print(summary)
print(cost_summary)

In [ ]:
labels = [CONDITION_LABELS.get(c, c) for c in summary["condition"]]
acc_means = summary["test_accuracy_mean"].values
acc_stds = summary["test_accuracy_std"].fillna(0).values
f1_means = summary["test_macro_f1_mean"].values
f1_stds = summary["test_macro_f1_std"].fillna(0).values

x = range(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - width/2 for i in x], acc_means, width, yerr=acc_stds, capsize=3, label="Accuracy")
ax.bar([i + width/2 for i in x], f1_means, width, yerr=f1_stds, capsize=3, label="Macro-F1")
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.0)
ax.set_title("Test-set performance by condition")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR + "results_figure_overall.png")
plt.show()

In [ ]:
gap_conditions = ["monolingual_de", "multilingual_in_language_de", "multilingual_zero_shot_transfer"]
gap_df = summary[summary["condition"].isin(gap_conditions)].copy()
gap_df["condition"] = pd.Categorical(gap_df["condition"], categories=gap_conditions, ordered=True)
gap_df = gap_df.sort_values("condition")

gap_labels = [CONDITION_LABELS[c] for c in gap_df["condition"]]
gap_acc = gap_df["test_accuracy_mean"].values
gap_f1 = gap_df["test_macro_f1_mean"].values

x = range(len(gap_labels))
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar([i - width/2 for i in x], gap_acc, width, label="Accuracy", color="#25344F")
ax.bar([i + width/2 for i in x], gap_f1, width, label="Macro-F1", color="#CC9E4C")
ax.set_xticks(list(x))
ax.set_xticklabels(gap_labels, rotation=15, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.0)
ax.set_title("German performance: in-language vs. zero-shot")
ax.legend()
if len(gap_acc) == 3:
    gap_value = gap_acc[1] - gap_acc[2]
    ax.annotate(f"gap = {gap_value:.3f}", xy=(2, gap_acc[2] + 0.03), ha="center", fontsize=10)
fig.tight_layout()
fig.savefig(OUTPUT_DIR + "results_figure_zeroshot_gap.png")
plt.show()

In [ ]:
cost_plot_df = cost_summary[~cost_summary["condition"].isin(["baseline_en", "baseline_de"])]
cost_labels = [CONDITION_LABELS.get(c, c) for c in cost_plot_df["condition"]]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(cost_labels, cost_plot_df["train_seconds"], color="#839958")
axes[0].set_ylabel("Seconds")
axes[0].set_title("Training time by condition")
axes[0].set_xticks(range(len(cost_labels)))
axes[0].set_xticklabels(cost_labels, rotation=30, ha="right")

axes[1].bar(cost_labels, cost_plot_df["peak_memory_mb"], color="#D3968C")
axes[1].set_ylabel("MB")
axes[1].set_title("Peak GPU memory by condition")
axes[1].set_xticks(range(len(cost_labels)))
axes[1].set_xticklabels(cost_labels, rotation=30, ha="right")

fig.tight_layout()
fig.savefig(OUTPUT_DIR + "results_figure_cost.png")
plt.show()

In [ ]:
latex_lines = []
for _, row in summary.iterrows():
    label = CONDITION_LABELS.get(row["condition"], row["condition"])
    acc_mean, acc_std = row["test_accuracy_mean"], row["test_accuracy_std"]
    f1_mean, f1_std = row["test_macro_f1_mean"], row["test_macro_f1_std"]
    acc_str = f"{acc_mean:.3f} $\\pm$ {acc_std:.3f}" if pd.notna(acc_std) else f"{acc_mean:.3f}"
    f1_str = f"{f1_mean:.3f} $\\pm$ {f1_std:.3f}" if pd.notna(f1_std) else f"{f1_mean:.3f}"
    latex_lines.append(f"{label} & {acc_str} & {f1_str} \\\\")

with open(OUTPUT_DIR + "results_table.tex", "w") as f:
    f.write("\n".join(latex_lines))
print("\n".join(latex_lines))